In [2]:
import torch
x = torch.zeros(4,8)

In [3]:
x.numel()

32

In [4]:
a = torch.tensor([1e-8], dtype = torch.float16)
assert a == 0

In [5]:
a = torch.tensor([1e-8], dtype = torch.bfloat16)
assert a != 0

In [6]:
torch.finfo(torch.float32)

finfo(resolution=1e-06, min=-3.40282e+38, max=3.40282e+38, eps=1.19209e-07, smallest_normal=1.17549e-38, tiny=1.17549e-38, dtype=float32)

In [7]:
torch.finfo(torch.float16)

finfo(resolution=0.001, min=-65504, max=65504, eps=0.000976562, smallest_normal=6.10352e-05, tiny=6.10352e-05, dtype=float16)

In [8]:
torch.finfo(torch.bfloat16)

finfo(resolution=0.01, min=-3.38953e+38, max=3.38953e+38, eps=0.0078125, smallest_normal=1.17549e-38, tiny=1.17549e-38, dtype=bfloat16)

In [9]:
from einops import *

x = torch.ones(3,4)
y = torch.ones(4,3)

# Old way
z = x @ y

In [10]:
# New way

z_einops = einsum(x, y, "seq1 hidden, hidden seq2 -> seq1 seq2")

In [11]:
z == z_einops

tensor([[True, True, True],
        [True, True, True],
        [True, True, True]])

In [12]:
x = torch.ones(2,3,4)
y = torch.ones(2,3,4)

In [13]:
z_old = x @ y.transpose(-2,-1)

In [14]:
z_ei = einsum(x,y, "batch seq1 hidden, batch seq2 hidden -> batch seq1 seq2" )

In [15]:
x = torch.ones(3,8)
x = rearrange(x, "... (heads hidden1) -> ... heads hidden1", heads = 2)
x.shape

torch.Size([3, 2, 4])

In [21]:
B = 1024
D = 256
K = 64
device = "cuda"
x = torch.ones(B,D, device=device)
w = torch.ones(D,K, device= device)

In [50]:
y = x @ w

How Many flops? 
A single scalar product takes D mul and D-1 sum, suppose D 2 * D, then I have to do it for every columns and for every row, so total flops = 2 * D * B* K

In [102]:
import timeit

B = 4096
D = 4096
K = 4096
actual_num_flops = 2 * D * B * K
x = torch.ones(B,D, device=device,dtype=torch.bfloat16)
w = torch.ones(D,K, device= device,dtype=torch.bfloat16)
def run():
   x @ w 
   torch.cuda.synchronize()
num_trials = int(10)
total_time = timeit.timeit(run, number = num_trials)
actual_time = total_time/num_trials
actual_flop_per_sec = actual_num_flops / actual_time
actual_flop_per_sec

3960378379571.6504

In [61]:
actual_num_flops = 2 * D * B * K

In [62]:
actual_flop_per_sec = actual_num_flops / actual_time

In [63]:
actual_flop_per_sec

273797416893.00964

In [10]:
# Params accounting 
def params_account(V, N, D, F):

   P = N * (4 * D**2 + 2 * D + 3* D *F) + 2* V* D + D
   return P



In [11]:
#GPT2 xl
params_gpt2_xl = params_account(50257, 48, 1600, 4288)
params_gpt2_xl


1640452800

In [13]:
x = params_gpt2_xl
print(f"{x:e}")

1.640453e+09


In [14]:
# in fp32 4 bytes for each parameter
bytes = 4 * x
GBib = (bytes / ( 2 ** 30))
print(f"{GBib:e}")



6.111163e+00


In [16]:
def return_flops(D, T, V, N, F):
   block = 8 * T * D**2 + 4 * T**2 * D + 6 * T * D * F
   full  = N * block + 2 * T * D * V
   return full

In [18]:
FLOPs_gpt2_xl = return_flops(1600, 1024, 50257, 48, 4288)
print(f"{FLOPs_gpt2_xl:e}")

3.516770e+12


In [32]:
def decompose_flops(D, T, V, N, F):
    FFN_cost = 6 * T * D * F * N
    Final_projection_layer = 2 * T * D * V
    mha = N * (8 * T * D**2 + 4 * T**2 * D)

    total_cost = return_flops(D, T, V, N, F)

    components = {
        "FFN": FFN_cost,
        "Final projection layer": Final_projection_layer,
        "MHA": mha,
    }

    print(f"Total FLOPs: {total_cost:,.0f}")
    print("-" * 65)

    for name, cost in components.items():
        percentage = 100 * cost / total_cost
        print(f"{name:<30} {cost:>22,.0f}  ({percentage:6.2f}%)")

    print("-" * 65)

    accounted_for = sum(components.values())

    print(
        f"{'Total accounted for':<30} "
        f"{accounted_for:>22,.0f}  "
        f"({100 * accounted_for / total_cost:6.2f}%)"
    )

    return components, total_cost



In [ ]:
models = {
    "GPT-2 Small": {
        "D": 768,
        "T": 1024,
        "V": 50_257,
        "N": 12,
        "F": round((8 / 3 * 768) / 64) * 64,
    },
    "GPT-2 Medium": {
        "D": 1024,
        "T": 1024,
        "V": 50_257,
        "N": 24,
        "F": round((8 / 3 * 1024) / 64) * 64,
    },
    "GPT-2 Large": {
        "D": 1280,
        "T": 1024,
        "V": 50_257,
        "N": 36,
        "F": round((8 / 3 * 1280) / 64) * 64,
    },
    "GPT-2 XL": {
        "D": 1600,
        "T": 1024,
        "V": 50_257,
        "N": 48,
        "F": 4288,
    },
}


for model_name, cfg in models.items():
   print("\n" + "=" * 70)
   print(model_name)
   print("=" * 70)

   print(
        f"D={cfg['D']}, "
        f"T={cfg['T']}, "
        f"V={cfg['V']:,}, "
        f"N={cfg['N']}, "
        f"F={cfg['F']}"
   )

   decompose_flops(
        D=cfg["D"],
        T=cfg["T"],
        V=cfg["V"],
        N=cfg["N"],
        F=cfg["F"],
   )


GPT-2 Small
D=768, T=1024, V=50,257, N=12, F=2048.0
Total FLOPs: 291,648,307,200
-----------------------------------------------------------------
FFN                                   115,964,116,992  ( 39.76%)
Final projection layer                 79,047,426,048  ( 27.10%)
MHA                                    96,636,764,160  ( 33.13%)
-----------------------------------------------------------------
Total accounted for                   291,648,307,200  (100.00%)

GPT-2 Medium
D=1024, T=1024, V=50,257, N=24, F=2752.0
Total FLOPs: 830,172,299,264
-----------------------------------------------------------------
FFN                                   415,538,085,888  ( 50.05%)
Final projection layer                105,396,568,064  ( 12.70%)
MHA                                   309,237,645,312  ( 37.25%)
-----------------------------------------------------------------
Total accounted for                   830,172,299,264  (100.00%)

GPT-2 Large
D=1280, T=1024, V=50,257, N=36, F=339

In [33]:
# --------------------------------------------------
# GPT-2 XL configuration
# --------------------------------------------------

D = 1600
V = 50_257
N = 48
F = 4288

context_lengths = [1024, 16_384]

results = {}

for T in context_lengths:
   print("\n" + "=" * 70)
   print(f"GPT-2 XL, context length T = {T:,}")
   print("=" * 70)

   components, total = decompose_flops(
        D=D,
        T=T,
        V=V,
        N=N,
        F=F,
   )

   results[T] = {
        "components": components,
        "total": total,
   }


# --------------------------------------------------
# Compare T = 1024 with T = 16384
# --------------------------------------------------

old_total = results[1024]["total"]
new_total = results[16_384]["total"]

factor = new_total / old_total
percent_increase = 100 * (new_total - old_total) / old_total

print("\n" + "=" * 70)
print("CHANGE IN TOTAL FLOPs")
print("=" * 70)

print(f"T = 1,024:  {old_total:,.0f} FLOPs")
print(f"T = 16,384: {new_total:,.0f} FLOPs")

print(f"\nIncrease factor: {factor:.2f}x")
print(f"Percentage increase: {percent_increase:.2f}%")


GPT-2 XL, context length T = 1,024
Total FLOPs: 3,516,769,894,400
-----------------------------------------------------------------
FFN                                 2,023,332,249,600  ( 57.53%)
Final projection layer                164,682,137,600  (  4.68%)
MHA                                 1,328,755,507,200  ( 37.78%)
-----------------------------------------------------------------
Total accounted for                 3,516,769,894,400  (100.00%)

GPT-2 XL, context length T = 16,384
Total FLOPs: 133,577,729,638,400
-----------------------------------------------------------------
FFN                                32,373,315,993,600  ( 24.24%)
Final projection layer              2,634,914,201,600  (  1.97%)
MHA                                98,569,499,443,200  ( 73.79%)
-----------------------------------------------------------------
Total accounted for               133,577,729,638,400  (100.00%)

CHANGE IN TOTAL FLOPs
T = 1,024:  3,516,769,894,400 FLOPs
T = 16,384: 133,577,